# Getting set up

Everything in this course lives in git. By the end of this notebook you will have

- git and a GitHub account, talking to each other over SSH,
- Python, Jupyter and the course's packages, managed by `uv`,
- and **two** repositories wired to one folder on your computer:

```
  materials  ──▶   the course repository.  Read-only to you.
                   New notes and assignments arrive here.
                   You pull from it.  You never push to it.

  origin     ◀──   your repository.  Private; yours and mine.
                   Your work lives here.
                   Pushing to it IS how you hand work in.
```

**Read sections 1–4 in your web browser.** You cannot run this notebook yet — section 4
is what puts it on your computer. From section 5 onward you will be running these cells
yourself, in Jupyter.

Take your time. None of this is hard, but it is the fiddliest thing you will do all
semester. Everything after it is four commands you will type until they are automatic.

## 1. Git and a GitHub account

macOS ships git as part of Apple's command line tools. Open **Terminal** and run

```bash
xcode-select --install
```

If it is already installed you will be told so, which is fine. Confirm with

```bash
git --version
```

Next, [create a GitHub account](https://docs.github.com/en/get-started/start-your-journey/creating-an-account-on-github)
if you do not already have one. Pick a username that you'd be comfortable with an employer seeing.  Use an email address you will still have after you graduate.

**Send me your username** — I need it to create your repository.

Finally, tell git who you are, so that your commits carry your name
([GitHub's instructions](https://docs.github.com/en/get-started/git-basics/set-up-git)):

```bash
git config --global user.name "Your Name"
git config --global user.email "you@example.com"
```

Use the same email as your GitHub account, or your commits will not be linked to your
profile. This is the one place the address actually matters — the one in the next
section is only a label.

## 2. An SSH key

GitHub stopped accepting passwords in 2021. You authenticate with an *SSH key* instead:
a pair of files, one secret and one public. GitHub gets the public one. The secret one
never leaves your laptop.

You do this once, and then never think about it again.

GitHub's own walkthroughs are good, and I would rather you follow them than my
paraphrase:

1. [Generate a new SSH key](https://docs.github.com/en/authentication/connecting-to-github-with-ssh/generating-a-new-ssh-key-and-adding-it-to-the-ssh-agent)
2. [Add it to your GitHub account](https://docs.github.com/en/authentication/connecting-to-github-with-ssh/adding-a-new-ssh-key-to-your-github-account)
3. [Test the connection](https://docs.github.com/en/authentication/connecting-to-github-with-ssh/testing-your-ssh-connection)

The short version, for macOS. Generate the key:

```bash
ssh-keygen -t ed25519 -C "your-name-laptop-or-some-other-informative-string"
```

The `-C` part is only a *comment* stored alongside the key. It plays no part in
authenticating you — GitHub identifies the key by its fingerprint — so it does not need
to match your GitHub email, and need not be an email at all. GitHub does offer it as the
default title when you paste the key in, which is why something naming the **machine**
is more useful than an address: when you come to revoke the key for a laptop you no
longer own, that is the label you will wish you had written.

Press Return at every prompt to accept the defaults. You may set a passphrase or leave
it empty; if you set one, the next step stops you having to type it constantly.

```bash
cat >> ~/.ssh/config <<'EOF'

Host github.com
  AddKeysToAgent yes
  UseKeychain yes
  IdentityFile ~/.ssh/id_ed25519
EOF

ssh-add --apple-use-keychain ~/.ssh/id_ed25519
```

`UseKeychain` is the macOS-specific part: it stores your passphrase in the login
keychain so you are never asked for it again. Skip it and you will be prompted in every
new terminal window, and you will conclude that git is broken. It is not.

Now copy the **public** half to your clipboard

```bash
pbcopy < ~/.ssh/id_ed25519.pub
```

and paste it into GitHub under *[Settings → SSH and GPG keys](https://github.com/settings/keys) → New SSH key*.

Note the `.pub`. Never paste the other file, `~/.ssh/id_ed25519`, anywhere at all —
that one is the secret, and it is the whole point.

Test it:

```bash
ssh -T git@github.com
```

Success looks like this:

```
Hi your-username! You've successfully authenticated, but GitHub does not provide shell access.
```

That message *is* success, despite sounding like a refusal. If instead you see
`Permission denied (publickey)`, work through
[GitHub's troubleshooting page](https://docs.github.com/en/authentication/troubleshooting-ssh/error-permission-denied-publickey).

## 3. `uv`

We manage Python with [`uv`](https://docs.astral.sh/uv/): one tool that installs Python
itself, builds the virtual environment, and installs packages — quickly, and identically
for everyone. You do not need to install Python separately, and you should *not* use the
`python3` that comes with macOS.

[Install it](https://docs.astral.sh/uv/getting-started/installation/):

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Close Terminal, open a new one, and check:

```bash
uv --version
```

We come back to `uv` in section 5, once you have the course files.

## 4. Your two repositories

I have created a private repository for you at

```
github.com/evanberkowitz/UVI-PHY320-2026-08-<your-github-username>
```

GitHub will email you an invitation; it also shows up at
[github.com/notifications](https://github.com/notifications). Accept it. The repository
is empty right now — you are about to fill it.

Run these commands, replacing `<your-github-username>` with yours:

```bash
# first cd into a directory where you want your coursework to live.  Then:
git clone https://github.com/evanberkowitz/UVI-PHY320-2026-08-materials.git phy320
cd phy320
git remote rename origin materials
git remote add origin git@github.com:evanberkowitz/UVI-PHY320-2026-08-<your-github-username>.git
git push -u origin main
```

Read what you just did, because those five lines are the whole architecture:

- **`git clone`** copies the course repository — notes, assignments, this notebook —
  into a new folder called `phy320`. Cloning automatically names the place it came from
  `origin`.
- **`git remote rename origin materials`** renames that. By convention `origin` means
  *your* repository, and the place you cloned from is not yours. Calling it `materials`
  keeps the two straight in your head and in your commands.
- **`git remote add origin …`** points `origin` at the private repository I made for
  you. Note that it is a `git@github.com:` address rather than `https://` — that is the
  SSH form, which is why you made the key in section 2.
- **`git push -u origin main`** uploads everything to your repository. The `-u` records
  that from now on, a bare `git push` means "push to `origin`".

You now have one folder with two remotes attached to it. Confirm:

```bash
git remote -v
```

You should see `materials` pointing at an `https://…-materials.git` address, and
`origin` pointing at `git@…-<your-username>.git`. If `origin` still points at
`-materials`, you have missed a step. ([More on remotes.](https://docs.github.com/en/get-started/getting-started-with-git/managing-remote-repositories))

### Why clone the course repository instead of starting empty?

Because it gives your repository and the course repository a **shared history**. Git can
only merge two histories sensibly when it can find a common ancestor between them.
Had you started from an empty folder and copied files in, git would see two unrelated
histories, and every update I published would be a fight. This way `git pull materials
main` simply works, all semester.

## 5. Python packages and Jupyter

From inside the `phy320` folder:

```bash
uv sync
```

That reads `pyproject.toml` and `uv.lock` and installs the exact versions of NumPy,
SciPy, Matplotlib, Jupyter and everything else the course uses, into a virtual
environment inside the folder. Because `uv.lock` pins exact versions, your environment
matches mine and your classmates'. When something misbehaves, it will not be because you
have a different SciPy. ([More on uv projects.](https://docs.astral.sh/uv/guides/projects/))

Now start Jupyter:

```bash
uv run jupyter lab
```

A browser tab opens. Open `note/getting-set-up.ipynb` — this notebook — and carry on
from here **in Jupyter, on your own machine**.

`uv run` is worth remembering: it runs a command inside the project's environment,
without you having to "activate" anything first.

## 6. Two things about notebooks and git

A `.ipynb` file is JSON, and its *outputs* — every number printed, every figure,
as base64 — are stored in it too. Both facts make notebooks awkward in git, and
both have a one-time fix. Everything you need arrived with `uv sync`.

**Outputs are not committed.** Your working copy keeps them; what git records is
the code you wrote. Switch that on once, from inside `phy320`:

```bash
git config filter.nbstripout.clean "uv run nbstripout"
git config filter.nbstripout.smudge cat
git config filter.nbstripout.required true
```

Two reasons this matters. Outputs are where notebook merge conflicts come from,
so stripping them keeps `git pull` from becoming an unreadable mess every time I
revise an assignment. And your *code* is what is being marked — I run your
notebook rather than reading a picture of what it printed on your laptop.

The `required true` part means git will refuse to commit a notebook if this is
not working, rather than quietly committing your outputs. If `git add` ever
complains about a notebook, that is this, and re-running the three lines above is
the fix.

**Diffs are readable.** [`nbdime`](https://nbdime.readthedocs.io/en/latest/)
teaches git to compare notebooks cell by cell instead of as JSON:

```bash
uv run nbdime config-git --enable
```

Now `git diff` on a notebook shows changed *cells*, and you can compare two
notebooks side by side in a browser:

```bash
uv run nbdiff-web note/getting-set-up.ipynb note/binary.ipynb
```

You will want that later, when solutions are released and you compare mine
against yours. ([nbdime's git integration.](https://nbdime.readthedocs.io/en/latest/vcs.html))

## 7. Check your setup

Run the next four cells. Each prints `✓` for something that works, or `✗` together with
the command that fixes it. They only look at things; nothing here changes anything.

If something fails, fix it and run the cell again.

If you cannot get to all `✓`, bring this output to office hours. Pasting it tells me
far more than "git isn't working".

In [ ]:
# Run this cell first. It defines the checking machinery used by the cells below.

import importlib
import re
import shutil
import subprocess
import sys

MATERIALS = "UVI-PHY320-2026-08-materials"
PASS, FAIL = "✓", "✗"


def run(*command):
    # Run a command; return (returncode, combined output). Never raises.
    try:
        finished = subprocess.run(command, capture_output=True, text=True, timeout=30)
        return finished.returncode, (finished.stdout + finished.stderr).strip()
    except FileNotFoundError:
        return 127, command[0] + ": command not found"
    except subprocess.TimeoutExpired:
        return 124, command[0] + ": timed out"


def check(label, ok, detail="", fix=""):
    # Print one result line, and the fix underneath it if the check failed.
    print((PASS if ok else FAIL) + " " + label + ("  " + detail if detail else ""))
    if not ok and fix:
        for line in fix.strip().splitlines():
            print("      " + line.strip())
    return ok

In [ ]:
# --- git, and who GitHub thinks you are ------------------------------------

code, version = run("git", "--version")
check("git is installed", code == 0, version,
      "Run:  xcode-select --install")

code, name = run("git", "config", "--global", "--get", "user.name")
check("git knows your name", code == 0 and bool(name), name,
      'Run:  git config --global user.name "Your Name"')

code, email = run("git", "config", "--global", "--get", "user.email")
check("git knows your email", code == 0 and bool(email), email,
      'Run:  git config --global user.email "you@example.com"')

# `ssh -T` exits non-zero even when it succeeds, so read the message, not the code.
code, reply = run("ssh", "-T",
                  "-o", "BatchMode=yes",
                  "-o", "StrictHostKeyChecking=accept-new",
                  "-o", "ConnectTimeout=10",
                  "git@github.com")
greeting = re.search(r"Hi ([^!]+)!", reply)
check("SSH authenticates you to GitHub", greeting is not None,
      "as " + greeting.group(1) if greeting else reply.splitlines()[0] if reply else "",
      "Redo section 2, then run:  ssh -T git@github.com\n"
      "https://docs.github.com/en/authentication/troubleshooting-ssh/error-permission-denied-publickey")

In [ ]:
# --- your two repositories -------------------------------------------------

code, toplevel = run("git", "rev-parse", "--show-toplevel")
inside = code == 0
check("you are inside the course repository", inside, toplevel,
      "Quit Jupyter, cd into the phy320 folder you cloned, and run:  uv run jupyter lab")

if inside:
    _, remotes = run("git", "remote", "-v")

    materials = re.search(r"^materials\s+(\S+)", remotes, re.M)
    check("remote 'materials' points at the course",
          materials is not None and MATERIALS in materials.group(1),
          materials.group(1) if materials else "",
          "Run:  git remote rename origin materials")

    origin = re.search(r"^origin\s+(\S+)", remotes, re.M)
    check("remote 'origin' is your own repository",
          origin is not None and MATERIALS not in origin.group(1),
          origin.group(1) if origin else "",
          "Run:  git remote add origin git@github.com:evanberkowitz/"
          "UVI-PHY320-2026-08-<your-github-username>.git")

    check("origin uses SSH rather than HTTPS",
          origin is not None and origin.group(1).startswith("git@"), "",
          "Run:  git remote set-url origin git@github.com:evanberkowitz/"
          "UVI-PHY320-2026-08-<your-github-username>.git")

    code, tracking = run("git", "rev-parse", "--abbrev-ref",
                         "--symbolic-full-name", "@{upstream}")
    check("a bare 'git push' knows where to go",
          code == 0 and tracking == "origin/main",
          tracking if code == 0 else "not set yet",
          "Run:  git push -u origin main")

In [ ]:
# --- Python, the packages, and the tools -----------------------------------

check("Python is new enough", sys.version_info >= (3, 11), sys.version.split()[0],
      "Quit Jupyter and restart it with:  uv run jupyter lab")

for package in ("numpy", "scipy", "matplotlib", "nbdime", "nbstripout"):
    try:
        importlib.import_module(package)
        installed, note = True, ""
    except ImportError:
        installed, note = False, "not importable"
    check(package + " is available", installed, note,
          "Run:  uv sync\nthen restart Jupyter with:  uv run jupyter lab")

check("uv is installed", shutil.which("uv") is not None, "",
      "See https://docs.astral.sh/uv/getting-started/installation/")

code, cleaner = run("git", "config", "--get", "filter.nbstripout.clean")
check("notebook outputs are kept out of git", code == 0 and "nbstripout" in cleaner,
      "", "Run the three `git config filter.nbstripout...` lines in section 6")

code, driver = run("git", "config", "--get", "diff.jupyternotebook.command")
check("git can diff notebooks readably", code == 0 and "nbdime" in driver, "",
      "Run:  uv run nbdime config-git --enable")

## 8. The weekly rhythm

Once you are set up, the whole semester is four commands.

**Starting an assignment** — collect whatever I have published since you last looked:

```bash
git pull materials main
```

**While you work** — commit whenever you get a piece working. Name whichever notebook
you are actually working on; the one below is only an example:

```bash
git add assignment/03-odes/odes.ipynb
git commit -m "pendulum right-hand side, and one RK4 step"
```

**When you want me to see it**, including when you are finished:

```bash
git push
```

That push *is* the submission. There is no other button, no upload form, no email.

I grade the last commit you pushed before the deadline, so anything pushed is safe.
Push early and push often: a laptop that dies at 11pm the night before then costs you
nothing.

Two habits worth building from week one. Commit when a piece works, rather than once at
the end — it gives you a point to retreat to when the next change breaks everything.
And write commit messages that say what actually changed, so that `git log` reads as an
account of your thinking. Both are marked eventually, and both are simply what the
professional habit looks like.

After each deadline, my solutions appear in a `solution/` folder on your next
`git pull`. They arrive alongside your work and never touch your files.

## 9. When something goes wrong

**`Permission denied (publickey)`** — SSH cannot prove who you are. Re-run the test from
section 2, `ssh -T git@github.com`, and work through
[GitHub's troubleshooting page](https://docs.github.com/en/authentication/troubleshooting-ssh/error-permission-denied-publickey).

**`Updates were rejected because the remote contains work that you do not have locally`** —
I have pushed something to your repository, usually a released solution, since you last
pulled. Collect it first, then push:

```bash
git pull
git push
```

**`fatal: not a git repository`** — you are in the wrong folder. `cd` into `phy320`.

**A merge conflict inside a notebook** — git could not decide between two versions of a
cell. Do not panic, and do not delete anything:

```bash
git status                          # which files are conflicted
uv run nbdiff                       # look at what actually differs
```

Then edit the cells to what you want them to be, `git add` the file, and `git commit`.
Bring the first one to office hours. Resolving conflicts is a skill, not a disaster, and
you will meet it again in every job you ever have.

**`Your branch is ahead of 'origin/main' by 3 commits`** — you committed but never
pushed. Your work exists only on your laptop, and I cannot see it. `git push`.

---

Once every check in section 7 shows `✓`, you are ready. Your first assignment is
`assignment/00-git/introduction-to-git.ipynb`.